# 抽屉原理 完整教程：从直觉到严格证明

## 📚 教学目标
1. 理解**抽屉原理 (Pigeonhole Principle)** 的基本形式和推广形式
2. 掌握**袜子配对**问题中的最坏情况分析
3. 学会**握手问题**中识别「鸽子」和「巢」
4. 理解**几何应用**中如何构造子区域作为「巢」
5. 区分抽屉原理的**确定性保证**与生日问题的**概率估计**

**参考书**：《A Practical Guide to Quantitative Finance Interviews》(Xinfeng Zhou) 第2章
**教学策略**：先用最简单的例子建立直觉，再逐步引入更精妙的「鸽子-巢」构造

---

## 1. 抽屉原理的基本形式

### 🎯 原理陈述

**基本形式**：如果 $n+1$ 只鸽子要飞进 $n$ 个巢，则至少有一个巢里有 2 只或更多鸽子。

$$n+1 \text{ 鸽子} \rightarrow n \text{ 巢} \implies \exists \text{ 巢包含} \geq 2 \text{ 只鸽子}$$

**推广形式**：如果 $kn+1$ 只鸽子飞进 $n$ 个巢，则至少有一个巢里有 $k+1$ 只或更多鸽子。

$$\lceil m/n \rceil \text{ 是某个巢中鸽子数的下界}$$

### 💡 为什么这么有用？

抽屉原理的威力在于**存在性证明**：
- 不需要找出具体是哪个巢
- 不需要考虑概率
- 给出**绝对保证** —— 在最坏情况下也成立

关键是识别问题中的「鸽子」和「巢」。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from scipy.spatial.distance import pdist

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

In [ ]:
# ========== 步骤 1: 抽屉原理的直观演示 ==========
print("📊 步骤 1: 抽屉原理的直观演示")
print("─" * 60)

np.random.seed(42)

def pigeonhole_demo(n_pigeons, n_holes, n_trials=10000):
    """模拟鸽子随机飞入巢, 统计最大占用数"""
    max_counts = []
    for _ in range(n_trials):
        holes = np.zeros(n_holes, dtype=int)
        for _ in range(n_pigeons):
            hole = np.random.randint(0, n_holes)
            holes[hole] += 1
        max_counts.append(holes.max())
    return np.array(max_counts)

print(f"  n 只鸽子飞入 n-1 个巢 (保证至少有一个巢 >= 2):")
print()

for n in [3, 5, 10, 20]:
    results = pigeonhole_demo(n, n - 1, 10000)
    pct_ge2 = (results >= 2).mean() * 100
    avg_max = results.mean()
    print(f"  {n} 鸽子 → {n-1} 巢: 最大占用 >= 2 的比例 = {pct_ge2:.1f}% (平均最大 = {avg_max:.2f})")

print(f"\n  💡 无论鸽子如何分配, 最大占用数总是 >= 2!")
print(f"     这不是概率结论, 而是逻辑必然。模拟只是验证。")

In [ ]:
# ========== 可视化: 鸽子分配示意图 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 三种分配方案: 5 鸽子 → 4 巢
np.random.seed(42)
allocations = [
    [2, 1, 1, 1],  # 最均匀
    [3, 1, 1, 0],  # 中等
    [5, 0, 0, 0],  # 最不均匀
]
titles = ['Most Uniform: [2,1,1,1]', 'Medium: [3,1,1,0]', 'Most Skewed: [5,0,0,0]']

for ax, alloc, title in zip(axes, allocations, titles):
    colors = ['#2ecc71' if a >= 2 else 'steelblue' for a in alloc]
    bars = ax.bar(range(1, 5), alloc, color=colors, edgecolor='black', alpha=0.85)
    
    for bar, v in zip(bars, alloc):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                   str(v), ha='center', va='bottom', fontweight='bold', fontsize=14)
    
    ax.axhline(y=2, color='#e74c3c', linestyle='--', linewidth=1.5, 
              label='Pigeonhole bound (>= 2)')
    ax.set_xlabel('Hole Number', fontsize=12)
    ax.set_ylabel('Pigeons', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 5))
    ax.set_ylim(0, 6)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('5 Pigeons into 4 Holes: At Least One Hole Has >= 2', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色柱 = 包含 >= 2 只鸽子的巢 (符合抽屉原理预测)")
print(f"  无论如何分配, 至少有一个绿色柱 —— 这就是抽屉原理的保证")

---

## 2. 袜子配对问题 (Matching Socks)

### 🎯 问题描述

抽屉里有 $k$ 种颜色的袜子（每种颜色有很多只）。在完全黑暗中，至少要取出多少只袜子才能**保证**有一双颜色相同的？

### 📐 分析

- 鸽子 = 取出的袜子
- 巢 = 颜色种类（$k$ 种）

根据抽屉原理：取出 $k+1$ 只袜子，至少有 2 只颜色相同。

$$\text{最少取出数} = k + 1$$

### 💡 最坏情况思维

注意「保证」= 最坏情况。前 $k$ 只可能恰好每种颜色一只（Murphy's Law），第 $k+1$ 只一定和之前某只重复。

In [ ]:
# ========== 步骤 2: 袜子配对 —— 手算小例子 ==========
print("📊 步骤 2: 袜子配对问题")
print("─" * 60)

print("\n  例题: 抽屉有 4 种颜色的袜子 (红/蓝/绿/黄)")
print("  问: 至少取多少只保证有一双同色?")
print()
print("  📐 分析:")
print("    鸽子 = 取出的袜子")
print("    巢 = 颜色种类 = 4")
print("    最坏情况: 前 4 只恰好是 红/蓝/绿/黄 各一只")
print("    第 5 只不管什么颜色, 一定和前面某只重复")
print("    答案: k+1 = 4+1 = 5 只")

print("\n  推广: 至少取多少只保证有 m 双同色?")
print("    需要 k(m-1) + 1 只")
print("    例: 4 色, 保证 3 只同色 → 4×2+1 = 9 只")

In [ ]:
# ========== 步骤 3: Monte Carlo 验证袜子问题 ==========
np.random.seed(42)

def simulate_socks(n_colors, n_draws, n_sim=50000):
    """模拟取袜子, 返回有配对的比例"""
    pair_found = 0
    for _ in range(n_sim):
        drawn = np.random.randint(0, n_colors, n_draws)
        _, counts = np.unique(drawn, return_counts=True)
        if counts.max() >= 2:
            pair_found += 1
    return pair_found / n_sim

k = 10  # 10 种颜色

print(f"📊 步骤 3: 袜子配对 Monte Carlo 验证 (k={k} 种颜色)")
print(f"─" * 60)
print(f"\n  {'取出数':>8} {'有配对概率':>12} {'保证?':>8}")
print(f"  {'─'*8} {'─'*12} {'─'*8}")

for n_draws in range(1, k + 3):
    prob = simulate_socks(k, n_draws)
    guaranteed = "✅ 100%" if n_draws >= k + 1 else ""
    print(f"  {n_draws:8d} {prob:12.4f} {guaranteed:>8}")

print(f"\n  💡 取 {k+1} 只时概率跳到 100% —— 这就是抽屉原理的保证!")
print(f"     注意 k={k} 只时概率已经很高但不是 100%。")
print(f"     '几乎确定' 和 '保证' 是本质不同的!")

In [ ]:
# ========== 可视化: 配对概率随取出数的变化 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 不同颜色数下的配对概率曲线 ---
ax1 = axes[0]
for k_val, color in [(5, '#2ecc71'), (10, 'steelblue'), (20, '#e67e22')]:
    draws_range = range(1, k_val + 5)
    probs = [simulate_socks(k_val, d, 20000) for d in draws_range]
    ax1.plot(list(draws_range), probs, 'o-', color=color, linewidth=2, 
             markersize=6, label=f'k={k_val} colors')
    ax1.axvline(x=k_val + 1, color=color, linestyle='--', alpha=0.4)

ax1.axhline(y=1.0, color='#e74c3c', linestyle='-', linewidth=1.5, alpha=0.5)
ax1.set_xlabel('Number of Socks Drawn', fontsize=12)
ax1.set_ylabel('P(At Least One Pair)', fontsize=12)
ax1.set_title('Probability of Finding a Matching Pair', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_ylim(-0.05, 1.1)

# --- 右图: 最坏情况的取出序列示例 ---
ax2 = axes[1]
k_demo = 4
# 最坏情况: 前 k 只每种一只, 第 k+1 只重复
worst_case = list(range(k_demo)) + [0]  # 颜色编号
color_names = ['Red', 'Blue', 'Green', 'Yellow']
bar_colors = ['#e74c3c', 'steelblue', '#2ecc71', '#e67e22', '#e74c3c']

x_pos = range(1, len(worst_case) + 1)
bars = ax2.bar(x_pos, [1]*len(worst_case), color=bar_colors, edgecolor='black', alpha=0.85)

for i, (pos, c) in enumerate(zip(x_pos, worst_case)):
    label = color_names[c]
    ax2.text(pos, 0.5, label, ha='center', va='center', fontsize=10, fontweight='bold',
            rotation=45)

# 标注配对
ax2.annotate('', xy=(5, 1.15), xytext=(1, 1.15),
            arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=2.5))
ax2.text(3, 1.25, 'PAIR!', ha='center', fontsize=14, fontweight='bold', color='#e74c3c')

ax2.set_xlabel('Draw Number', fontsize=12)
ax2.set_ylabel('', fontsize=12)
ax2.set_title(f'Worst Case: {k_demo} Colors, Need {k_demo+1} Draws', 
              fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_ylim(0, 1.5)
ax2.set_yticks([])
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：配对概率随取出数增加而上升, 在 k+1 处保证 100%")
print(f"  右图：4 种颜色的最坏情况 —— 前 4 只各不同, 第 5 只必定重复")

---

## 3. 握手问题 (Handshakes)

### 🎯 问题描述

一个派对上有 $n$ 个人。证明：至少有两个人握了**相同次数**的手。

### 📐 证明

每个人可能的握手次数范围是 $0$ 到 $n-1$（与 0 到 $n-1$ 个其他人握手）。

看起来有 $n$ 种可能的值 $\{0, 1, 2, \ldots, n-1\}$ 给 $n$ 个人分配 —— 似乎可以各不相同？

**关键**：$0$ 和 $n-1$ 不可能同时出现！
- 如果有人握了 $n-1$ 次手（和所有人都握了），就没人可以是 0 次
- 如果有人是 0 次（和谁都没握），就没人可以是 $n-1$ 次

所以实际只有 $n-1$ 种可能值，但有 $n$ 个人。

$$n \text{ 人 (鸽子)} \rightarrow n-1 \text{ 种可能的握手数 (巢)} \implies \text{至少两人握手次数相同}$$

In [ ]:
# ========== 步骤 4: 握手问题 —— 手算小例子 ==========
print("📊 步骤 4: 握手问题 —— 小规模枚举")
print("─" * 60)

print("\n  例: n=4 人 (A, B, C, D)")
print("  可能的握手次数: 0, 1, 2, 3")
print("  但 0 和 3 不共存! 所以只有:")
print("    情况 1: 包含 0 → 可能值 {0, 1, 2} → 3 种值给 4 人")
print("    情况 2: 包含 3 → 可能值 {1, 2, 3} → 3 种值给 4 人")
print("  两种情况都是 4 人 → 3 种值 → 必有重复!")

print("\n  所有可能的握手图 (n=4, 无自环, 无向):")
people = ['A', 'B', 'C', 'D']
n = 4
all_edges = list(combinations(range(n), 2))  # 所有可能的边

collision_count = 0
total_graphs = 0

# 枚举所有子图
for n_edges in range(len(all_edges) + 1):
    for edges in combinations(all_edges, n_edges):
        total_graphs += 1
        degrees = [0] * n
        for u, v in edges:
            degrees[u] += 1
            degrees[v] += 1
        
        if len(set(degrees)) < n:  # 有重复度数
            collision_count += 1

print(f"\n  总共可能的握手图数: {total_graphs}")
print(f"  存在重复握手次数的图数: {collision_count}")
print(f"  比例: {collision_count/total_graphs*100:.1f}%")
print(f"\n  💡 100%! 每种握手图都至少有两人握手次数相同!")

In [ ]:
# ========== 步骤 5: Monte Carlo 验证 (大规模) ==========
np.random.seed(42)

def simulate_handshakes(n_people, n_sim=20000):
    """随机生成握手图, 检查是否有两人握手次数相同"""
    collision_count = 0
    for _ in range(n_sim):
        # 随机决定每对人是否握手
        degrees = np.zeros(n_people, dtype=int)
        for i in range(n_people):
            for j in range(i + 1, n_people):
                if np.random.random() < 0.5:
                    degrees[i] += 1
                    degrees[j] += 1
        
        if len(set(degrees)) < n_people:
            collision_count += 1
    
    return collision_count / n_sim

print(f"📊 步骤 5: 握手问题 Monte Carlo 验证")
print(f"─" * 60)
print(f"\n  {'人数':>6} {'有重复的比例':>14} {'理论':>8}")
print(f"  {'─'*6} {'─'*14} {'─'*8}")

for n_people in [3, 5, 8, 10, 15, 20]:
    prob = simulate_handshakes(n_people, 10000)
    print(f"  {n_people:6d} {prob:14.4f} {'100%':>8}")

print(f"\n  💡 所有模拟结果都是 100%!")
print(f"     抽屉原理的结论是确定性的, 不依赖于概率。")

In [ ]:
# ========== 可视化: 握手次数分布 ==========
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n_ppl in zip(axes, [6, 10, 20]):
    # 生成一个随机握手图
    degrees = np.zeros(n_ppl, dtype=int)
    for i in range(n_ppl):
        for j in range(i + 1, n_ppl):
            if np.random.random() < 0.5:
                degrees[i] += 1
                degrees[j] += 1
    
    # 找出重复的度数
    unique, counts = np.unique(degrees, return_counts=True)
    duplicated_vals = set(unique[counts > 1])
    
    colors = ['#e74c3c' if d in duplicated_vals else 'steelblue' for d in degrees]
    
    bars = ax.bar(range(1, n_ppl + 1), sorted(degrees), color=[
        '#e74c3c' if d in duplicated_vals else 'steelblue' 
        for d in sorted(degrees)], edgecolor='black', alpha=0.85)
    
    ax.set_xlabel('Person (sorted by handshakes)', fontsize=11)
    ax.set_ylabel('Handshake Count', fontsize=11)
    ax.set_title(f'n={n_ppl}: Handshake Counts (red = duplicated)', 
                fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # 标注
    dup_str = f'Duplicates: {dict(zip(unique[counts>1], counts[counts>1]))}'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.98, 0.98, dup_str, transform=ax.transAxes, fontsize=9,
           verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  红色柱 = 握手次数有重复的人 (至少两人同数)")
print(f"  每个图中都一定有红色柱 —— 抽屉原理的保证")
print(f"  人数越多, 重复越多 (可能值只有 n-1 种, 但有 n 人)")

---

## 4. 蚂蚁在正方形 (Points in a Square)

### 🎯 问题描述

在边长为 1 的正方形内放置 5 个点。证明：至少有两个点之间的距离不超过 $\frac{\sqrt{2}}{2} \approx 0.707$。

### 📐 证明

将单位正方形分成 4 个**相等的小正方形**（边长 $1/2$）：

```
┌─────┬─────┐
│  1  │  2  │
├─────┼─────┤
│  3  │  4  │
└─────┴─────┘
```

- 鸽子 = 5 个点
- 巢 = 4 个小正方形

由抽屉原理，至少有 2 个点在同一个小正方形内。

小正方形的对角线长度 = $\sqrt{(1/2)^2 + (1/2)^2} = \frac{\sqrt{2}}{2}$

同一个小正方形内任意两点的最大距离 = 对角线长 = $\frac{\sqrt{2}}{2}$

$$\therefore \text{至少两点距离} \leq \frac{\sqrt{2}}{2} \approx 0.707 \qquad \blacksquare$$

In [ ]:
# ========== 步骤 6: 蚂蚁问题 —— Monte Carlo 验证 ==========
np.random.seed(42)

N_SIM = 50000
N_POINTS = 5
threshold = np.sqrt(2) / 2

min_distances = []
for _ in range(N_SIM):
    points = np.random.uniform(0, 1, (N_POINTS, 2))
    dists = pdist(points)  # 所有点对的距离
    min_distances.append(dists.min())

min_dist_arr = np.array(min_distances)

print(f"📊 步骤 6: 蚂蚁在正方形 —— Monte Carlo 验证")
print(f"─" * 60)
print(f"  正方形边长: 1")
print(f"  点数: {N_POINTS}")
print(f"  阈值: sqrt(2)/2 = {threshold:.6f}")
print(f"  模拟次数: {N_SIM:,}")
print(f"")
print(f"  最小距离 <= sqrt(2)/2 的比例: {(min_dist_arr <= threshold).mean()*100:.2f}%")
print(f"  最小距离的平均值: {min_dist_arr.mean():.4f}")
print(f"  最小距离的最大值: {min_dist_arr.max():.4f}")
print(f"  (理论上界: {threshold:.4f})")
print(f"")
print(f"  💡 100% 的情况下最小距离 <= sqrt(2)/2 !")
print(f"     而且实际的最小距离通常远小于理论上界。")

In [ ]:
# ========== 可视化: 点在正方形中的分布 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

np.random.seed(42)

for idx, ax in enumerate(axes[:2]):
    # 生成 5 个随机点
    np.random.seed(42 + idx * 17)
    points = np.random.uniform(0, 1, (5, 2))
    
    # 画子正方形网格
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    
    # 标注子正方形
    for i, (cx, cy) in enumerate([(0.25, 0.75), (0.75, 0.75), (0.25, 0.25), (0.75, 0.25)]):
        ax.text(cx, cy, f'Cell {i+1}', ha='center', va='center', fontsize=9, 
               alpha=0.3, fontweight='bold')
    
    # 确定每个点所在的子正方形
    cells = []
    for px, py in points:
        cell = int(px >= 0.5) + 2 * int(py < 0.5)
        cells.append(cell)
    
    # 找重复的 cell
    cell_counts = {}
    for c in cells:
        cell_counts[c] = cell_counts.get(c, 0) + 1
    crowded_cells = {c for c, cnt in cell_counts.items() if cnt >= 2}
    
    # 画点
    for i, ((px, py), cell) in enumerate(zip(points, cells)):
        color = '#e74c3c' if cell in crowded_cells else '#2ecc71'
        ax.scatter(px, py, s=120, c=color, edgecolors='black', linewidth=1.5, zorder=5)
        ax.annotate(f'P{i+1}', (px, py), textcoords="offset points", 
                   xytext=(8, 8), fontsize=10, fontweight='bold')
    
    # 画最近点对之间的连线
    dists = pdist(points)
    min_idx = np.argmin(dists)
    # 将压缩距离索引转换为点对索引
    pair_idx = 0
    for i in range(5):
        for j in range(i+1, 5):
            if pair_idx == min_idx:
                ax.plot([points[i,0], points[j,0]], [points[i,1], points[j,1]], 
                       'r-', linewidth=2.5, zorder=4)
                mid_x = (points[i,0] + points[j,0]) / 2
                mid_y = (points[i,1] + points[j,1]) / 2
                ax.text(mid_x, mid_y + 0.05, f'd={dists.min():.3f}', 
                       ha='center', fontsize=10, color='#e74c3c', fontweight='bold')
            pair_idx += 1
    
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.set_title(f'Example {idx+1}: min dist = {dists.min():.3f} <= {threshold:.3f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.grid(alpha=0.2)

# --- 右图: 最小距离分布 ---
ax3 = axes[2]
ax3.hist(min_dist_arr, bins=50, color='steelblue', edgecolor='black', alpha=0.85, density=True)
ax3.axvline(x=threshold, color='#e74c3c', linestyle='--', linewidth=2.5,
           label=f'sqrt(2)/2 = {threshold:.3f}')
ax3.set_xlabel('Minimum Pairwise Distance', fontsize=12)
ax3.set_ylabel('Density', fontsize=12)
ax3.set_title('Distribution of Min Distance (5 Points in Unit Square)',
             fontsize=12, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左/中图：5 个点在单位正方形中, 虚线分出 4 个子正方形")
print(f"          红色点 = 同一子正方形中的点 (距离一定 <= sqrt(2)/2)")
print(f"  右图：50000 次模拟中, 最小距离始终 < sqrt(2)/2 (红线)")

---

## 5. 生日问题：抽屉原理 vs 概率

### 🎯 问题描述

一年有 365 天。
- **抽屉原理保证**：至少需要 **366** 人才能**保证**有两人同一天生日
- **概率视角**：只需要 **23** 人，就有 **>50%** 的概率有两人同一天生日

### 📐 概率计算

$n$ 人中**没有**生日冲突的概率：

$$P(\text{no collision}) = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365} \cdots \frac{365-n+1}{365} = \frac{365!}{(365-n)! \cdot 365^n}$$

$$P(\text{collision}) = 1 - P(\text{no collision})$$

### 💡 两种思维的区别

| 方法 | 结论 | 含义 |
|------|------|------|
| 抽屉原理 | 366 人保证冲突 | 100% 确定性，无例外 |
| 概率分析 | 23 人有 50% 概率 | 统计意义上的可能性 |

In [ ]:
# ========== 步骤 7: 生日问题概率计算 ==========
print("📊 步骤 7: 生日问题 —— 概率 vs 抽屉原理")
print("─" * 60)

def birthday_prob(n, days=365):
    """n 人中至少两人同一天生日的概率"""
    if n > days:
        return 1.0
    p_no_collision = 1.0
    for i in range(n):
        p_no_collision *= (days - i) / days
    return 1 - p_no_collision

print(f"\n  {'人数':>6} {'碰撞概率':>12} {'备注'}")
print(f"  {'─'*6} {'─'*12} {'─'*20}")

milestones = [5, 10, 15, 20, 23, 30, 40, 50, 57, 70, 100, 365, 366]
for n in milestones:
    p = birthday_prob(n)
    note = ""
    if n == 23:
        note = "← 超过 50%!"
    elif n == 57:
        note = "← 超过 99%"
    elif n == 366:
        note = "← 抽屉原理: 100%"
    print(f"  {n:6d} {p:12.6f} {note}")

print(f"\n  💡 23 人就有 > 50% 概率! 57 人有 > 99% 概率!")
print(f"     但只有 366 人才是 100% 确定 (抽屉原理)。")

In [ ]:
# ========== 步骤 8: Monte Carlo 验证生日问题 ==========
np.random.seed(42)

N_SIM = 100000

def simulate_birthday(n_people, n_sim, days=365):
    collisions = 0
    for _ in range(n_sim):
        birthdays = np.random.randint(0, days, n_people)
        if len(set(birthdays)) < n_people:
            collisions += 1
    return collisions / n_sim

print(f"📊 步骤 8: Monte Carlo 验证")
print(f"  模拟次数: {N_SIM:,}")
print(f"─" * 60)
print(f"\n  {'人数':>6} {'理论概率':>12} {'模拟概率':>12} {'误差':>10}")
print(f"  {'─'*6} {'─'*12} {'─'*12} {'─'*10}")

for n in [10, 20, 23, 30, 50, 70]:
    theory = birthday_prob(n)
    sim = simulate_birthday(n, N_SIM)
    error = abs(theory - sim)
    print(f"  {n:6d} {theory:12.6f} {sim:12.6f} {error:10.6f}")

print(f"\n  💡 模拟结果与理论值高度吻合!")

In [ ]:
# ========== 可视化: 生日问题概率曲线 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 概率曲线 ---
ax1 = axes[0]
n_range = range(1, 101)
probs = [birthday_prob(n) for n in n_range]

ax1.plot(n_range, probs, color='steelblue', linewidth=2.5, label='Exact probability')
ax1.axhline(y=0.5, color='#e74c3c', linestyle='--', linewidth=1.5, label='P = 0.5')
ax1.axvline(x=23, color='#e67e22', linestyle='--', linewidth=1.5, label='n = 23')
ax1.axhline(y=1.0, color='#2ecc71', linestyle='-', linewidth=1.5, alpha=0.5,
           label='Pigeonhole (n=366)')

# 标注 50% 交叉点
ax1.scatter([23], [birthday_prob(23)], s=100, c='#e67e22', zorder=5, edgecolors='black')
ax1.annotate(f'n=23, P={birthday_prob(23):.3f}', xy=(23, birthday_prob(23)),
            xytext=(35, 0.35), fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))

ax1.set_xlabel('Number of People (n)', fontsize=12)
ax1.set_ylabel('P(At Least One Shared Birthday)', fontsize=12)
ax1.set_title('Birthday Problem: Probability vs. n', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(alpha=0.3)

# --- 右图: 抽屉原理 vs 概率 对比 ---
ax2 = axes[1]

# 不同 "天数" 下的 50% 阈值 和 抽屉原理阈值
days_list = [7, 30, 100, 365, 1000]
prob_50 = []
pigeon_threshold = []

for d in days_list:
    # 找 50% 概率需要的人数
    for n in range(1, d + 2):
        if birthday_prob(n, d) >= 0.5:
            prob_50.append(n)
            break
    pigeon_threshold.append(d + 1)

x_pos = np.arange(len(days_list))
width = 0.35

bars1 = ax2.bar(x_pos - width/2, prob_50, width, color='steelblue', 
                edgecolor='black', alpha=0.85, label='50% Probability')
bars2 = ax2.bar(x_pos + width/2, pigeon_threshold, width, color='#e74c3c',
                edgecolor='black', alpha=0.85, label='Pigeonhole (100%)')

for bar, v in zip(bars1, prob_50):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(v), ha='center', va='bottom', fontweight='bold', fontsize=10)

for bar, v in zip(bars2, pigeon_threshold):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(v), ha='center', va='bottom', fontweight='bold', fontsize=10)

ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'd={d}' for d in days_list])
ax2.set_ylabel('Number of People Needed', fontsize=12)
ax2.set_title('50% Probability vs. 100% Guarantee', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：生日碰撞概率随人数快速增长, 23 人就超过 50%")
print(f"  右图：50% 概率需要 ~sqrt(d) 人, 但 100% 保证需要 d+1 人")
print(f"        差距巨大! 这就是概率思维 vs 最坏情况思维的区别")

---

## 6. 整除性应用 (Divisibility)

### 🎯 问题描述

从 $\{1, 2, \ldots, 2n\}$ 中任取 $n+1$ 个整数。证明：其中必有两个数是相邻的（相差 1）。

### 📐 证明

构造 $n$ 个「巢」：将 $\{1, 2, \ldots, 2n\}$ 分成 $n$ 对相邻数：

$$\{1, 2\}, \{3, 4\}, \{5, 6\}, \ldots, \{2n-1, 2n\}$$

- 鸽子 = $n+1$ 个选出的整数
- 巢 = $n$ 对相邻数

由抽屉原理，$n+1$ 个数放入 $n$ 个「巢」中，至少有一个「巢」包含 2 个数。

同一个「巢」中的两个数恰好相邻（相差 1）。$\blacksquare$

### 💡 推广

类似地，可以证明从 $\{1, \ldots, 2n\}$ 中取 $n+1$ 个数，必有一个整除另一个。
- 巢的构造：$\{1, 2, 4, 8, \ldots\}$, $\{3, 6, 12, \ldots\}$, $\{5, 10, 20, \ldots\}$, ... (按奇数部分分组)

In [ ]:
# ========== 步骤 9: 相邻整数问题 —— 枚举验证 ==========
print("📊 步骤 9: 相邻整数问题 (小规模枚举)")
print("─" * 60)

n = 4  # 从 {1,...,8} 中取 5 个数
universe = list(range(1, 2*n + 1))
k = n + 1  # 取 5 个

print(f"  从 {{1, ..., {2*n}}} 中取 {k} 个数")
print(f"  巢: {[(2*i-1, 2*i) for i in range(1, n+1)]}")
print()

total = 0
has_consecutive = 0

for subset in combinations(universe, k):
    total += 1
    s = sorted(subset)
    found = False
    for i in range(len(s) - 1):
        if s[i+1] - s[i] == 1:
            found = True
            break
    if found:
        has_consecutive += 1

print(f"  C({2*n},{k}) = {total} 种选法")
print(f"  包含相邻数的: {has_consecutive}")
print(f"  比例: {has_consecutive/total*100:.1f}%")
print(f"\n  💡 100%! 每种选法都包含相邻整数!")

In [ ]:
# ========== 步骤 10: Monte Carlo 大规模验证 ==========
np.random.seed(42)

print("📊 步骤 10: 相邻整数问题 —— Monte Carlo 验证")
print("─" * 60)

N_SIM = 50000

print(f"\n  {'n':>6} {'集合':>20} {'取数':>6} {'有相邻比例':>12}")
print(f"  {'─'*6} {'─'*20} {'─'*6} {'─'*12}")

for n in [5, 10, 20, 50, 100]:
    universe = np.arange(1, 2*n + 1)
    k = n + 1
    
    count_consec = 0
    for _ in range(N_SIM):
        chosen = sorted(np.random.choice(universe, k, replace=False))
        for i in range(len(chosen) - 1):
            if chosen[i+1] - chosen[i] == 1:
                count_consec += 1
                break
    
    print(f"  {n:6d} {{1,...,{2*n}}}{'':>{14-len(str(2*n))}} {k:6d} {count_consec/N_SIM*100:12.1f}%")

print(f"\n  💡 所有规模下都是 100%! 抽屉原理的结论不受规模影响。")

In [ ]:
# ========== 可视化: 整除性抽屉构造 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 相邻数对的「巢」构造 ---
ax1 = axes[0]
n_vis = 6
pairs = [(2*i-1, 2*i) for i in range(1, n_vis+1)]

# 随机选 n+1=7 个数
np.random.seed(42)
chosen = sorted(np.random.choice(range(1, 2*n_vis+1), n_vis+1, replace=False))

for i, (a, b) in enumerate(pairs):
    y = n_vis - i
    # 画巢 (方框)
    rect = plt.Rectangle((0, y-0.3), 3, 0.6, fill=False, edgecolor='gray', linewidth=1.5)
    ax1.add_patch(rect)
    
    # 画数字
    for j, num in enumerate([a, b]):
        x = 0.75 + j * 1.5
        if num in chosen:
            color = '#e74c3c'
            ax1.scatter(x, y, s=400, c=color, edgecolors='black', linewidth=2, zorder=5)
        else:
            color = 'lightgray'
            ax1.scatter(x, y, s=400, c=color, edgecolors='gray', linewidth=1, zorder=5)
        ax1.text(x, y, str(num), ha='center', va='center', fontsize=12, fontweight='bold')
    
    ax1.text(3.5, y, f'Pair {i+1}', va='center', fontsize=10, color='gray')

# 标注某个有两个被选中的巢
for i, (a, b) in enumerate(pairs):
    if a in chosen and b in chosen:
        y = n_vis - i
        rect = plt.Rectangle((0, y-0.35), 3, 0.7, fill=False, 
                            edgecolor='#e74c3c', linewidth=3)
        ax1.add_patch(rect)
        ax1.text(5, y, f'Consecutive!\n({a},{b})', va='center', fontsize=11, 
                color='#e74c3c', fontweight='bold')

ax1.set_xlim(-0.5, 7)
ax1.set_ylim(-0.5, n_vis + 0.5)
ax1.set_title(f'Pigeonhole: {n_vis+1} Numbers from {{1,...,{2*n_vis}}}',
             fontsize=13, fontweight='bold')
ax1.axis('off')

# --- 右图: 不同 n 下需要取多少数才保证有相邻 ---
ax2 = axes[1]
ns = np.arange(2, 51)
guarantee = ns + 1  # n+1

# 模拟: 不选 n+1, 只选 n 个, 有相邻的概率
probs_n = []
for n_val in ns:
    count = 0
    for _ in range(5000):
        chosen_test = sorted(np.random.choice(range(1, 2*n_val+1), n_val, replace=False))
        for j in range(len(chosen_test)-1):
            if chosen_test[j+1] - chosen_test[j] == 1:
                count += 1
                break
    probs_n.append(count / 5000)

ax2.plot(ns, probs_n, 'o-', color='steelblue', markersize=3, linewidth=1.5,
        label='P(consecutive) with n chosen')
ax2.axhline(y=1.0, color='#2ecc71', linestyle='-', linewidth=2, alpha=0.7,
           label='Guaranteed (n+1 chosen)')

ax2.set_xlabel('n', fontsize=12)
ax2.set_ylabel('Probability of Consecutive Pair', fontsize=12)
ax2.set_title('Choosing n vs n+1: Probability vs Guarantee', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：红色 = 被选中的数, 红框 = 同一'巢'中有两个 → 相邻")
print(f"  右图：选 n 个数时概率 < 100% (但通常很高), 选 n+1 个时保证 100%")

In [ ]:
# ========== 综合汇总报告 ==========
print("=" * 60)
print("📋 抽屉原理应用综合报告")
print("=" * 60)

print(f"\n🎯 核心思想:")
print(f"   如果 n+1 个对象分入 n 个类别, 至少有一个类别包含 >= 2 个对象。")
print(f"   关键是识别问题中的 '鸽子' (对象) 和 '巢' (类别)。")

print(f"\n📊 问题汇总:")
print(f"   {'问题':<16} {'鸽子':<14} {'巢':<18} {'结论'}")
print(f"   {'─'*16} {'─'*14} {'─'*18} {'─'*20}")
print(f"   {'袜子配对':<16} {'取出的袜子':<14} {'颜色种类(k)':<18} {'取k+1只保证配对'}")
print(f"   {'握手问题':<16} {'n个人':<14} {'n-1种可能次数':<18} {'至少两人同数'}")
print(f"   {'点在正方形':<16} {'5个点':<14} {'4个子正方形':<18} {'两点距<=√2/2'}")
print(f"   {'生日问题':<16} {'366人':<14} {'365天':<18} {'保证同一天生日'}")
print(f"   {'相邻整数':<16} {'n+1个数':<14} {'n对相邻数':<18} {'必有两数相邻'}")

print(f"\n🧮 应用技巧:")
print(f"   1. 识别 '鸽子': 待分配的对象 (人/点/数/袜子)")
print(f"   2. 构造 '巢': 巧妙的分类方式 (颜色/区间/子区域)")
print(f"   3. 验证: 鸽子数 > 巢数 → 至少一个巢有 >= 2 只鸽子")
print(f"   4. 推导: 同一个巢中的对象满足所需的性质")

print("\n" + "=" * 60)

---

## 7. 核心概念回顾

### 📌 抽屉原理 (Pigeonhole Principle)
- **定义**: $n+1$ 个对象放入 $n$ 个类别，至少一个类别有 $\geq 2$ 个对象
- **公式**: $m$ 个对象放入 $n$ 个类别，至少一个类别有 $\geq \lceil m/n \rceil$ 个对象
- **含义**: 给出存在性的绝对保证，不依赖概率
- **判断标准**: 对象数 > 类别数

### 📌 最坏情况分析 (Worst-Case Analysis)
- **定义**: 考虑所有可能情况中最不利的那个
- **应用**: 确定「保证」某结论所需的最小条件
- **含义**: 与平均情况、概率分析的本质区别

### 📌 巢的构造技巧
- **定义**: 将对象的取值空间巧妙划分为有限个类别
- **应用**: 几何问题分割子区域、数论问题分组整数
- **含义**: 同一类别中的对象天然满足某种性质（距离近、相邻、整除等）

### 📌 生日问题 (Birthday Problem)
- **定义**: $n$ 人中至少两人同一天生日的概率
- **公式**: $P = 1 - \prod_{i=0}^{n-1} \frac{365-i}{365}$
- **含义**: 概率视角 (23 人 > 50%) vs 确定性 (366 人 = 100%)

### 🔗 完整流程
```
分析问题 → 识别鸽子和巢 → 验证鸽子数 > 巢数 → 推导结论
    ↓            ↓                ↓                ↓
 明确目标    巧妙构造分类      应用抽屉原理    同巢性质推导
```

### 📝 问题模式汇总

| 问题类型 | 巢的构造 | 同巢性质 | 典型例子 |
|---------|---------|---------|--------|
| 匹配/重复 | 属性值 | 属性相同 | 袜子、生日 |
| 距离/邻近 | 空间子区域 | 距离有界 | 蚂蚁在正方形 |
| 数论性质 | 数对/等价类 | 相邻/整除 | 连续整数 |
| 图论性质 | 度数范围 | 度数相同 | 握手问题 |

---

## 8. 常见误区

### ❌ 误区 1: 抽屉原理只能证明「至少 2 个」
**✓ 正确理解**: 推广形式可以证明「至少 $k+1$ 个」—— 如果 $kn+1$ 个对象放入 $n$ 个巢，至少有一个巢包含 $k+1$ 个。例如，3 种颜色的袜子取 7 只，保证有 3 只同色。

### ❌ 误区 2: 生日问题中 23 人就「保证」有生日重复
**✓ 正确理解**: 23 人只是概率超过 50%，不是保证。100% 保证需要 366 人（抽屉原理）。「很可能」和「确定」是本质不同的概念。面试中要明确区分概率结论和确定性结论。

### ❌ 误区 3: 握手问题中 0 到 n-1 都可以取到
**✓ 正确理解**: 0（不和任何人握手）和 $n-1$（和所有人都握手）不能同时出现。这使得实际只有 $n-1$ 种可能的握手次数，从而让抽屉原理生效。发现这个约束是解题的关键。

### ❌ 误区 4: 蚂蚁问题中随便分割正方形就行
**✓ 正确理解**: 子区域的划分必须使同一区域内任意两点的距离有上界。等分成 4 个小正方形是最自然的选择，因为小正方形的对角线恰好是 $\sqrt{2}/2$。不恰当的分割可能无法给出紧的距离上界。

### ❌ 误区 5: 抽屉原理只是简单的计数论证
**✓ 正确理解**: 虽然原理本身简单，但难点在于**构造合适的「巢」**。好的巢需要满足：(1) 数量恰好比鸽子少, (2) 同巢对象具有所需性质。巢的构造往往需要深刻的洞察力，这正是面试考察的重点。